# Reasoning Demo

## 1. Setup & imports

In [1]:
from chromadb import PersistentClient
from dotenv import load_dotenv

from omics_rag_playground.reasoning import answer_question

load_dotenv();

/Users/ema/Documents/personal_projects/omics-rag-playground/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
import textwrap

def wrap(text: str, width: int = 100) -> str:
    lines = text.split("\n")
    wrapped = [textwrap.fill(line, width=width) if line.strip() else line for line in lines]
    return "\n".join(wrapped)

def pretty_print(result, question: str, width: int = 100) -> None:
    """Display a ReasoningResult in a notebook-friendly format.
    
    Parameters
    ----------
    result : ReasoningResult
        The output of reasoning.answer_question.
    question : str
        The original user question, printed alongside the result for context.
    width : int
        Max line width for wrapping the answer text.
    """
    
    print(f"Question: {wrap(question, width)}")
    print(f"Reasoning type: {result.reasoning_type}")
    print(f"Confidence: {result.confidence}")
    print()
    print("Answer:")
    print(wrap(result.answer, width))
    print()
    print(f"Citations: {', '.join(result.citations) if result.citations else '(none)'}")
    print()
    print("Retrieved (PMID, distance):")
    for pmid, dist in zip(result.retrieved_pmids, result.retrieved_distances):
        marker = "★" if pmid in (result.citations or []) else " "
        print(f"  {marker} {pmid}  d={dist:.3f}")

## 2. Load ChromaDB collection

In [3]:
client = PersistentClient(path="../data/processed/chroma_db")
collection = client.get_collection(name="pubmed_abstracts_no_mesh")
print(f"Collection contains {collection.count()} documents.")

Collection contains 64 documents.


## 3. Define the three demo questions

In [4]:
q1 = "Which genes are involved in EMT in CRC?"
q2 = "What is the role of BEST4 in CRC?"
q3 = "How does WNT signaling drive CRC progression through the top DE genes?"

### Query 1 — Topic — EMT in CRC
**Type**: topic — "which X are associated with Y"

**Expected**: list of genes from the retrieved abstracts associated with EMT in CRC, confidence moderate to high if the corpus contains EMT-relevant abstracts.

In [5]:
result1 = answer_question(q1, collection)
pretty_print(result1, q1)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6955.84it/s]


Question: Which genes are involved in EMT in CRC?
Reasoning type: topic
Confidence: none

Answer:
The provided abstracts do not contain information specifically addressing which genes are involved
in epithelial-mesenchymal transition (EMT) in colorectal cancer. While the abstracts identify
numerous genes associated with CRC tumorigenesis, progression, and prognosis (such as FOXQ1, CDH3,
CLDN1, AQP8, CA7, SHMT2, TRIP13, and others), they do not explicitly discuss EMT as a mechanism or
identify genes specifically involved in EMT processes in CRC.

Citations: (none)

Retrieved (PMID, distance):
    35114976  d=1.109
    29885515  d=1.123
    39644158  d=1.198
    32462020  d=1.203
    36576616  d=1.215


**Observed**: confidence `none`, no citations. The model honestly reports that the retrieved abstracts do not address EMT specifically, even though they mention CRC-related genes (FOXQ1, CDH3, CLDN1, AQP8, CA7, SHMT2, TRIP13).

**Why**: the corpus was built dynamically from gene-symbol queries on the top DE genes of Stage 1. Process-focused queries like "EMT in CRC" find no exact-topic match — the closest abstracts mention several genes but do not focus on EMT as a process. All retrieved distances above 1.10 quantify this gap and provide the signal Block 4 will use to short-circuit the LLM and return a "no relevant literature" fallback before paying for a generation.

### Query 2 — Function — BEST4 role

**Type**: function — "what does gene X do"

**Expected**: characterization of BEST4's role in CRC drawn from the retrieved abstracts, confidence moderate (single-domain question, expected to be well-supported).

In [6]:
result2 = answer_question(q2, collection)
pretty_print(result2, q2)

Question: What is the role of BEST4 in CRC?
Reasoning type: function
Confidence: moderate

Answer:
BEST4 has a complex and seemingly contradictory role in colorectal cancer (CRC) depending on the
context:

According to one study (35058597), BEST4 functions as an oncogene in CRC. BEST4 expression is
upregulated in clinical CRC samples, and high-level expression correlates with advanced TNM stage,
lymph node metastasis, and poor survival. Functionally, ectopic BEST4 expression promotes CRC cell
proliferation and metastasis. Mechanistically, BEST4 binds to the p85α regulatory subunit of PI3K
and promotes p110 kinase activity, leading to activation of Akt signaling and expression of MYC and
CCND1—critical regulators of cell proliferation and metastasis. BEST4-mediated proliferation and
metastasis can be repressed by Akt inhibition.

However, a more recent study (39699952) demonstrates the opposite role: BEST4 functions as a tumor
suppressor. BEST4 overexpression attenuates cell proliferati

### Query 3 — Mechanism — WNT signaling

**Type**: mechanism — "how does X drive Y"

**Expected**: the query is deliberately ill-formed — "the top DE genes" refers to context the system does not have access to (the Stage 1 DE results). Expected: partial answer with caveats and lower confidence.

In [7]:
result3 = answer_question(q3, collection)
pretty_print(result3, q3)

Question: How does WNT signaling drive CRC progression through the top DE genes?
Reasoning type: mechanism
Confidence: moderate

Answer:
Based on the provided abstracts, WNT signaling drives CRC progression through dysregulation of key
genes, particularly in the Wnt/β-catenin pathway. The top downregulated genes in colorectal adenomas
(CRA) include Wnt pathway antagonists such as SFRP2 and GREM2, while top upregulated genes include
WNT2, WNT5A, and DKK4. DMBT1 loss, identified as a significant event during CRC development,
activates the WNT/β-catenin pathway, which enhances organoid growth and cell migration. In duodenal
adenomas (which show similar molecular features to CRA), gene expression analysis demonstrated
upregulation of the Wnt/β-catenin pathway with β-catenin accumulation confirmed in 80% of cases.
Additionally, stage-IV salient biomarkers in colorectal cancer progression include DKK1, a Wnt
pathway regulator. However, the abstracts do not provide detailed mechanistic explan

**Observed**: confidence `moderate`, four citations. The model answers the WNT-CRC mechanism question well, but does **not** acknowledge that it has no way to identify "the top DE genes" — it answers as if the question were "how does WNT signaling drive CRC progression". This is a silent re-interpretation of the question rather than a flagged ambiguity.

Two things worth noting:
- The top-1 retrieved abstract (PMID 29885515, d=0.860 — by far the closest match) is **not cited**. The model correctly recognizes that low cosine distance does not imply relevance to the specific claim, and only cites abstracts that genuinely support the answer.
- A production-grade system would explicitly detect references to absent context ("the top DE genes") and ask for clarification or reject the query. 

## 4. Spot-check: citation accuracy

We pick the first PMID cited in Q2 and confront the model's claims with the abstract text. This is the minimal hand-verified accuracy check required by the Definition of Done, it confirms that the model is not hallucinating citations.

In [11]:
pmid = result2.citations[0]
abstract = collection.get(ids=[pmid])
print(f"Abstract for PMID {pmid}:")
print(wrap(abstract['documents'][0], width=100))

Abstract for PMID 35058597:
Oncogenic potential of BEST4 in colorectal cancer via activation of PI3K/Akt signaling.

BEST4 is a member of the bestrophin protein family that plays a critical role in human intestinal
epithelial cells. However, its role and mechanism in colorectal cancer (CRC) remain largely elusive.
Here, we investigated the role and clinical significance of BEST4 in CRC. Our results demonstrate
that BEST4 expression is upregulated in clinical CRC samples and its high-level expression
correlates with advanced TNM (tumor, lymph nodes, distant metastasis) stage, LNM (lymph node
metastasis), and poor survival. Functional studies revealed that ectopic expression of BEST4
promoted CRC cell proliferation and metastasis, whereas the depletion of BEST4 had the opposite
effect both in vitro and in vivo. Mechanistically, BEST4 binds to the p85α regulatory subunit of
phosphatidylinositol-3-kinase (PI3K) and promotes p110 kinase activity; this leads to activation of
Akt signaling an

**Verification of Q2 claims against PMID 39699952**:

- "BEST4 overexpression attenuates cell proliferation, colony formation, and mobility in CRC in vitro" — matches the abstract.
- "BEST4 impedes tumour growth and the liver metastasis in vivo" — matches the abstract.
- "BEST4 downregulates TWIST1, thereby inhibiting EMT" — matches the abstract.
- "Low BEST4 mRNA correlates with advanced disease and worse prognosis" — matches the abstract.

Citation accuracy verified for all four claims attributed to this PMID. No hallucinated facts.

## 5. Takeaways

**Structured output works as intended**. The model returns a `GroundedAnswer` with `answer`, `citations`, `confidence`, and `reasoning_type` on every call. Confidence calibrates to evidence strength (`none` when retrieval fails, `moderate` when evidence is partial or contradictory, no `high` observed in this run — defensible given the small dynamic corpus).

**Citation grounding is not cosmetic**. In Q3 the closest retrieved abstract (PMID 29885515, d=0.860) is not cited — the model recognizes that proximity in embedding space does not imply support for the specific claim. Conversely, Q2 cites two PMIDs that genuinely back contradictory positions, and the spot-check confirms zero hallucinated facts against PMID 39699952.

**Reasoning_type is imperfect on multi-clause queries**. Q2 was originally phrased "Is BEST4 a marker of normal colonic differentiation? What is its role in CRC?" — a question with a topic clause and a function clause. The model consistently classified it as `topic` (the first clause dominated), even after a prompt instruction to "classify based on what the question asks for, not what the abstracts contain". Q2 was rewritten to be unambiguously function. A production system would either decompose multi-clause questions or treat reasoning_type as a distribution rather than a single label.

**Corpus limits expose themselves clearly**. Q1 (EMT topic) returns confidence `none` with all retrieved distances above 1.10. This is the signal that motivates Block 4: a distance-based pre-LLM fallback that short-circuits the model when retrieval is uniformly weak, saving cost and avoiding an LLM-generated "I don't know" that looks like a failure even when it isn't.

**The mechanism question (Q3) silently re-interprets the user's intent**. "How does WNT signaling drive CRC progression through the top DE genes?" presupposes context the system does not have. The model answers a *related* question instead of flagging the ambiguity. This is a real limitation of the current scaffold — not addressed in Block 4, parked for future iterations.